In [ ]:
import requests
import os
import zipfile
from io import BytesIO
import time

output_dir= r"C:\Users\caioa\Downloads\CNJ\PENDENTES_ACIMA_15"
os.makedirs(output_dir, exist_ok=True)

tribunais = [
    "TJAC", "TJAL", "TJAM", "TJAP", "TJBA", "TJCE", "TJDFT", "TJES", "TJGO",
    "TJMA", "TJMG", "TJMS", "TJMT", "TJPA", "TJPB", "TJPE", "TJPI",
    "TJPR", "TJRJ", "TJRN", "TJRO", "TJRR", "TJRS", "TJSC", "TJSE",
    "TJSP", "TJTO"
]

base_url = "https://api-csvr.cloud.cnj.jus.br/download_csv"

for tj in tribunais:
    print(f"Baixando {tj}...")

    params = {
        "tribunal": tj,
        "indicador": "ind2_15", #ind1 = novos, ind2 = pendentes, ind2_15 = pendentes acima 15 anos
        "oj":"",
        "grau": "",
        "municipio": "",
        "procedimento": "",
        "ambiente": "csv_p",
    }

    response = requests.get(
        base_url,
        params=params,
        timeout=120
    )

    response.raise_for_status()

    zip_path = os.path.join(output_dir, f"{tj}.zip")

    for _attempt in range(3):
        try:
            with open(zip_path, "wb") as f:
                f.write(response.content)
            break
        except Exception as e:
            if _attempt < 2:
                time.sleep(2)
                os.makedirs(output_dir, exist_ok=True)
            else:
                raise

    # with zipFile.ZipFile(zip_path, "r") as zip_ref:
    #     zip_ref.extractall(output_dir)

    # print(f"Arquivos extraídos para {output_dir}")

In [ ]:
import pandas as pd
from pathlib import Path
import csv
import io
import os
import re
import tempfile
import unicodedata
import zipfile

CSV_FIELD_LIMIT = min(sys.maxsize, 2_147_483_647)

csv.field_size_limit(CSV_FIELD_LIMIT)

# output_dir= r"C:\Users\caioa\Downloads\CNJ\NOVOS"
output_dir= r"C:\Users\caioa\Downloads\CNJ\PENDENTES"

tribunais = [
    "TJAC", "TJAL", "TJAM", "TJAP", "TJBA", "TJCE", "TJDFT", "TJES", "TJGO",
    "TJMA", "TJMG", "TJMS", "TJMT", "TJPA", "TJPB", "TJPE", "TJPI",
    "TJPR", "TJRJ", "TJRN", "TJRO", "TJRR", "TJRS", "TJSC", "TJSE",
    "TJSP", "TJTO"
]

for tj in tribunais:
    zip_path = os.path.join(output_dir, f"{tj}.zip")
    print(f"Filtrando {zip_path}...")

    # Informe o caminho do arquivo ZIP exportado do CNJ.
    INPUT_ZIP = Path(f"{zip_path}")

    # O resultado será salvo na mesma pasta do ZIP original.
    OUTPUT_ZIP = INPUT_ZIP.with_name(
        f"{INPUT_ZIP.stem}_filtrado.zip"
    )

    # Empresas aceitas no polo passivo.
    # Os nomes devem corresponder às instituições especificadas.
    TARGET_PATTERNS = tuple(
        re.compile(pattern)
        for pattern in (
            r"^(?:BANCO )?ITAU UNIBANCO(?: S A| SA)?$",
            r"^(?:BANCO )?ITAUCARD(?: S A| SA)?$",
            r"^(?:BANCO )?ITAU CONSIGNADO(?: S A| SA)?$",
        )
    )


    # ============================================================
    # 2. PADRONIZAÇÃO DOS NOMES
    # ============================================================

    def normalize(value: str) -> str:
        """Remove acentos e padroniza letras e pontuação."""

        plain = (
            unicodedata.normalize("NFKD", value)
            .encode("ascii", "ignore")
            .decode()
        )

        return " ".join(
            re.sub(r"[^A-Z0-9]+", " ", plain.upper()).split()
        )


    # ============================================================
    # 3. IDENTIFICAÇÃO DAS EMPRESAS
    # ============================================================

    def is_target(value: str) -> bool:
        """Verifica se pelo menos uma empresa desejada é ré."""

        # Evita processamento desnecessário na maioria das linhas.
        upper_value = value.upper()

        if "ITAU" not in upper_value and "ITAÚ" not in upper_value:
            return False

        # Trata processos com múltiplas partes no polo passivo.
        parties = value.strip().strip("{}").split(",")

        return any(
            pattern.fullmatch(normalize(party))
            for party in parties
            for pattern in TARGET_PATTERNS
        )


    # ============================================================
    # 4. LEITURA, FILTRAGEM E GRAVAÇÃO
    # ============================================================

    def filter_cnj_zip(
        source_path: Path,
        destination_path: Path
    ) -> tuple[int, int]:
        """Filtra os registros diretamente do ZIP, linha a linha."""

        source_path = source_path.resolve()
        destination_path = destination_path.resolve()

        # Valida a entrada antes do processamento.
        if not source_path.is_file() or not zipfile.is_zipfile(source_path):
            raise ValueError(f"ZIP inválido: {source_path}")

        if source_path == destination_path:
            raise ValueError("Entrada e saída devem ser diferentes.")

        destination_path.parent.mkdir(parents=True, exist_ok=True)

        read_count = 0
        selected_count = 0

        # Evita disponibilizar arquivos incompletos em caso de erro.
        with tempfile.NamedTemporaryFile(
            dir=destination_path.parent,
            suffix=".zip",
            delete=False
        ) as temp:
            temp_path = Path(temp.name)

        try:
            with zipfile.ZipFile(source_path) as source, zipfile.ZipFile(
                temp_path,
                "w",
                compression=zipfile.ZIP_DEFLATED,
                compresslevel=6
            ) as destination:

                csv_files = [
                    item
                    for item in source.infolist()
                    if not item.is_dir()
                    and item.filename.lower().endswith(".csv")
                ]

                if not csv_files:
                    raise ValueError("O ZIP não contém arquivos CSV.")

                for file_index, item in enumerate(csv_files):

                    # Lê o CSV sem descompactá-lo no disco.
                    with source.open(item) as input_binary, io.TextIOWrapper(
                        input_binary,
                        encoding="utf-8-sig",
                        newline=""
                    ) as input_text:

                        reader = csv.reader(input_text, delimiter=";")
                        header = next(reader, None)

                        if not header:
                            raise ValueError(f"CSV vazio: {item.filename}")

                        normalized_header = [
                            normalize(column) for column in header
                        ]

                        if normalized_header.count("POLO PASSIVO") != 1:
                            raise ValueError(
                                f"Coluna Polo passivo inválida: {item.filename}"
                            )

                        passive_index = normalized_header.index("POLO PASSIVO")

                        output_name = (
                            f"{file_index + 1:02d}_"
                            f"{Path(item.filename).stem}_filtrado.csv"
                        )

                        # Grava apenas os registros selecionados.
                        with destination.open(
                            output_name, "w", force_zip64=True
                        ) as output_binary, io.TextIOWrapper(
                            output_binary,
                            encoding="utf-8-sig",
                            newline=""
                        ) as output_text:

                            writer = csv.writer(
                                output_text,
                                delimiter=";",
                                lineterminator="\n"
                            )

                            writer.writerow(header)

                            for row_number, row in enumerate(reader, start=2):

                                read_count += 1

                                if len(row) != len(header):
                                    raise ValueError(
                                        f"Colunas inconsistentes em "
                                        f"{item.filename}, linha {row_number}"
                                    )

                                if is_target(row[passive_index]):
                                    writer.writerow(row)
                                    selected_count += 1

            # Substitui a saída somente após finalizar a gravação.
            os.replace(temp_path, destination_path)

            return read_count, selected_count

        finally:
            temp_path.unlink(missing_ok=True)


    # ============================================================
    # 5. EXECUÇÃO AUTOMÁTICA
    # ============================================================

    read_count, selected_count = filter_cnj_zip(
        INPUT_ZIP,
        OUTPUT_ZIP
    )

    print(f"Registros analisados: {read_count:,}")
    print(f"Registros filtrados: {selected_count:,}")
    print(f"Arquivo gerado: {OUTPUT_ZIP.resolve()}")
    print(f"Tamanho: {OUTPUT_ZIP.stat().st_size / 1024:.1f} KB")

Filtrando C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAC.zip...
Registros analisados: 118,012
Registros filtrados: 217
Arquivo gerado: C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAC_filtrado.zip
Tamanho: 23.3 KB
Filtrando C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAL.zip...
Registros analisados: 525,811
Registros filtrados: 4,879
Arquivo gerado: C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAL_filtrado.zip
Tamanho: 289.7 KB
Filtrando C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAM.zip...
Registros analisados: 953,462
Registros filtrados: 8,617
Arquivo gerado: C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAM_filtrado.zip
Tamanho: 459.1 KB
Filtrando C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAP.zip...
Registros analisados: 141,639
Registros filtrados: 383
Arquivo gerado: C:\Users\caioa\Downloads\CNJ\PENDENTES\TJAP_filtrado.zip
Tamanho: 33.8 KB
Filtrando C:\Users\caioa\Downloads\CNJ\PENDENTES\TJBA.zip...
Registros analisados: 3,479,859
Registros filtrados: 20,605
Arquivo gerado: C:\Users\caioa\Downloads\CNJ\PEND